# Dog Pose Estimation

Uses DeepLabCut for dog pose estimation.

In [ ]:
import deeplabcut
import tensorflow as tf
from dlclibrary import download_huggingface_model

In [ ]:
download_huggingface_model("full_dog", "dog_model")

In [ ]:
results = deeplabcut.video_inference_superanimal(
    ["files/dogs1.mp4"],  # it accepts images too
    "superanimal_quadruped",
    model_name="hrnet_w32",
    detector_name="fasterrcnn_resnet50_fpn_v2"
)

In [ ]:
import numpy as np
import pandas as pd

def mouth_openness(slice):
    individuals = slice.columns.get_level_values("individuals").unique()

    results = {}

    for ind in individuals:
        ind_df = slice.xs(ind, level="individuals", axis=1)
        ind_df = ind_df.droplevel(0, axis=1)

        ux = ind_df[("upper_jaw", "x")]
        uy = ind_df[("upper_jaw", "y")]
        lx = ind_df[("lower_jaw", "x")]
        ly = ind_df[("lower_jaw", "y")]

        openness = np.sqrt((ux - lx) ** 2 + (uy - ly) ** 2)

        results[ind] = openness.mean() 

    return results

def cut_dataframe(df, fps, timestamp):
    frame_id = int(timestamp * fps)

    start = max(0, frame_id - 10)
    end = frame_id + 10

    slice = df.iloc[start:end]
    return slice


slice = cut_dataframe(results['files/dogs1.mp4'], 30, 2.5)

results = mouth_openness(slice)

print(results)

{'animal0': 4.2716312,
 'animal1': 6.179832,
 'animal2': 6.2346783,
 'animal3': 13.127031,
 'animal4': 9.619755,
 'animal5': 0.0,
 'animal6': 0.0,
 'animal7': 0.0,
 'animal8': 0.0,
 'animal9': 0.0}